# ACP 协议（Agent Client Protocol）· 编辑器 ↔ 智能体

这一课讲 **ACP**：让「编辑器 / IDE」和「你自己写的本地智能体进程」用同一套协议说话。
一句话类比 ——

> ACP 之于「编辑器 ↔ 智能体」，就等于 **LSP（Language Server Protocol）**
> 之于「编辑器 ↔ 语言服务器」。
>
> 编辑器只出界面（界面、快捷键、文件上下文），智能体进程干活，
> 两者之间靠 ACP 这个「插座标准」对接。于是**同一个智能体可以插到
> Zed / PyCharm / VS Code / Neovim 里复用**，不用为每个编辑器写一次适配。

## 先把三个「协议」分清（这是本课最容易混的地方）

| 协议 | 谁 ↔ 谁 | 解决什么 | 本仓对应 |
|---|---|---|---|
| **MCP** | 智能体 ↔ 工具服务 | 让 Agent 用上外部工具 / 数据 | `Agent/05_mcp/` |
| **ACP** | **编辑器 ↔ 智能体** | 让 Agent 长在编辑器界面里 | **本课** |
| **A2A** | 智能体 ↔ 智能体 | 多个 Agent 互相派活 | `07_protocols/02_A2A协议.ipynb` |

三个协议的**传输层也不一样**，这点常被忽略：

| 协议 | 传输层 | 报文格式 |
|---|---|---|
| ACP | **stdio**（编辑器起子进程，管道读写） | JSON-RPC 2.0，**一行一个 JSON 对象** |
| A2A | **HTTP** | JSON-RPC 2.0 over HTTP |
| MCP | stdio 或 HTTP（都行） | JSON-RPC 2.0 |

注意最后一列 —— **三个协议全是 JSON-RPC 2.0**。所以把 JSON-RPC 的报文形态
真正看懂一次，后面 A2A 那几节就是「换个传输层」而已。这也是本 notebook
第 3 节的地位：它是整章的地基。

## 本课合并了三个源文件

| 源文件 | 角色 | 在本文里的位置 |
|---|---|---|
| `01_acp智能体.py` | 课案**原版**（46 行，最短实现） | 第 1 节 |
| `01_acp智能体_jxsd.py` | 课案**完整版** · 实操：把 Deep Agent 暴露成 ACP 服务端 | 第 2 节 |
| `02_acp原理_jxsd.py` | 课案**完整版** · 原理：把 JSON-RPC 2.0 报文自己收一发 | 第 3 节 |

第 1 节给你骨架，第 2 节给你完整实现（并且诚实演示「本机没装包时怎么办」），
第 3 节把 `deepagents-acp` 封装掉的那层**撕开**，用纯标准库真的收发一遍报文。

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 第 2 节的「模型连通性自检」会真实调用 `.env` 里配置的大模型（一次、一句话） |
| 依赖 | 已装：`langchain` / `langgraph` / `deepagents`（本项目 venv） |
| 依赖（缺） | **未装**：`acp` + `deepagents-acp` —— 只有「真起 ACP stdio 服务端」才需要它俩 |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（根目录 `.env`，已配置） |
| 前置服务 | 无。**第 3 节「原理」只用标准库**（`asyncio` / `json` / `os` / `subprocess` / `sys`），离线也能跑出真实收发 |
| 预计耗时 | 约 5~15 秒（含一次模型调用 + 两个协议演示） |

### 想真跑 ACP 服务端，要装什么

```powershell
cd F:\ProGram\Python_Base
uv add deepagents-acp          # 会连带装上 acp（它是 deepagents-acp 的依赖）
# 或：uv add acp deepagents-acp
```

装完之后 **本 notebook 不用改一行**：第 2 节的依赖自检会把 `_ACP_READY` 置为 `True`，
自动从「降级演示」切到「真的 `run_agent(server)`」。

> ⚠️ 装了包以后，那一格会**真的起一个 stdio 服务端并阻塞**（它在等编辑器的
> JSON-RPC 请求）—— 这时请手动中断内核（Ctrl+C / 重启内核），
> 因为 notebook 里没有编辑器来跟它对话。

## 本节地图

```mermaid
graph TB
    S0["0. 环境引导 + 依赖自检<br/>deepagents ✓ / acp ✗ / deepagents-acp ✗"]
    S1["1. 课案原版 46 行<br/>create_deep_agent 造一个 agent"]
    S2["2. 实操：插上 ACP 插座<br/>AgentServerACP + run_agent → stdio"]
    S3["3. 原理：撕开封装<br/>自己收一发 JSON-RPC 2.0"]
    D1["演示 ① 内存管道<br/>asyncio.Queue 当传输层"]
    D2["演示 ② 真实子进程<br/>stdin/stdout 管道 = PyCharm 的启动方式"]
    S0 --> S1 --> S2 --> S3 --> D1 --> D2
```

**等价文本（裸 JupyterLab 不渲染 mermaid，看这里）**：

```text
0. 环境引导 + 依赖自检（deepagents ✓ / acp ✗ / deepagents-acp ✗）
        ↓
1. 课案原版 46 行：create_deep_agent 造一个 agent
        ↓
2. 实操：插上 ACP 插座 —— AgentServerACP(agent) + run_agent(server) → stdio
        ↓
3. 原理：撕开封装，自己收一发 JSON-RPC 2.0
        ↓
   演示 ① 内存管道（asyncio.Queue 当传输层）
        ↓
   演示 ② 真实子进程（stdin/stdout 管道 = PyCharm 起你脚本的方式）
```

| 小节 | 一句话 | 需要什么 |
|---|---|---|
| 0 | 找到仓库根 + 探测依赖三件套 | 无 |
| 1 | 46 行造出一个 Deep Agent | `deepagents` |
| 2 | 把它挂到 ACP 的 stdio 插座上 | `acp` + `deepagents-acp`（缺则降级演示） |
| 3 | 手工收发 JSON-RPC，看清「一行一个 JSON」 | **无（纯标准库）** |

### 与上下节的衔接

- **上一节**（`06_langfuse/`）：那是「观测」，看 Agent 干了什么。
- **本课第 3 节**：JSON-RPC 2.0 的收发 —— **下一课 `02_A2A协议.ipynb` 会原样复用**
  （A2A 只是把同一套报文搬到 HTTP 上，再加「Agent Card」做服务发现）。
- **再下一节**：A2A 的两个端（CrewAI 端 / DeepAgents 端）与互相通信。

## 0. 环境引导

notebook 的工作目录默认是**它自己所在的文件夹**（`Agent/07_protocols/`），
而本项目所有代码都写 `from config import settings`（`config.py` 在仓库根）。

所以第一格统一做一件事：**向上找到仓库根 → `chdir` 过去 → 塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 0.1 依赖自检：课案三件套里，后两件本机没装

这是本课的**第一个教学点**。ACP 实操需要三个包：

| 包 | 作用 | 本机 venv |
|---|---|---|
| `deepagents` | 造 Deep Agent（`create_deep_agent`） | ✅ 已装 |
| `acp` | 官方 Python SDK，提供 `run_agent()` | ❌ 缺 |
| `deepagents-acp` | 把 DeepAgents 接进 ACP 的适配器（`AgentServerACP`） | ❌ 缺 |

按本仓「**模块层面永远不许因为缺包而 import 失败**」的铁律（`Agent/README.md` 第 164 行），
所有第三方 import 都用 `try/except ImportError` 兜住，把「缺什么」记在布尔开关里，
等到真正要跑的时候再决定走真流程还是降级演示。

另外单独兜一层 `deepagents.backends.LocalShellBackend` —— 课案的 `test.py` 用它给
Agent 一个「能执行 shell 命令」的后端，它在 `deepagents 0.7.x` 里位于 `deepagents.backends`
子包，未必每个版本都在同一位置，所以和上面三个分开探测。

最后一个细节：**notebook 里没有 `__file__`**（源脚本里有）。本节第 2.2 小节要复刻
「把 ACP 服务端脚本注册进编辑器」的 `acp.json`，里面正好用到 `str(__file__)`
（原脚本的语义是「本文件就是那个 ACP 服务端」）。这里显式把它指到归档的
ACP 服务端脚本上，语义等价、路径也真实可用。

In [ ]:
import asyncio
import json

from config import settings

# ---------- 依赖探测：三件套 + 一个额外后端 ----------
_HAS_DEEPAGENTS = False
_HAS_ACP_SDK = False
_HAS_DEEPAGENTS_ACP = False
_IMPORT_ERRORS: list[str] = []

try:
    from deepagents import create_deep_agent

    _HAS_DEEPAGENTS = True
except ImportError as exc:  # pragma: no cover - 本机已装
    _IMPORT_ERRORS.append(f"deepagents：{exc}")

try:
    from acp import run_agent  # type: ignore[import-not-found]

    _HAS_ACP_SDK = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"acp：{exc}")

try:
    from deepagents_acp.server import AgentServerACP  # type: ignore[import-not-found]

    _HAS_DEEPAGENTS_ACP = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"deepagents-acp：{exc}")

# LocalShellBackend：让 Agent 能执行 shell 命令（inherit_env 见第 2.3 小节）
_HAS_LOCAL_SHELL_BACKEND = False
try:
    from deepagents.backends import LocalShellBackend

    _HAS_LOCAL_SHELL_BACKEND = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"deepagents.backends.LocalShellBackend：{exc}")

# 真流程的开关：三件套齐了才走 run_agent(server)，否则降级
_ACP_READY = _HAS_DEEPAGENTS and _HAS_ACP_SDK and _HAS_DEEPAGENTS_ACP

# notebook 没有 __file__：把它指到归档的原版 ACP 服务端脚本（它模块级暴露了 agent）
__file__ = str(ROOT / "Agent" / "_py_source" / "07_protocols" / "01_acp智能体.py")

print("【依赖自检】deepagents:", _HAS_DEEPAGENTS,
      "| acp:", _HAS_ACP_SDK,
      "| deepagents-acp:", _HAS_DEEPAGENTS_ACP,
      "| LocalShellBackend:", _HAS_LOCAL_SHELL_BACKEND)
print("           能否真起 ACP stdio 服务端：", "能" if _ACP_READY else "不能（第 2 节走降级演示）")
print("           __file__ =", __file__)

### 预期输出

```text
【依赖自检】deepagents: True | acp: False | deepagents-acp: False | LocalShellBackend: True
           能否真起 ACP stdio 服务端： 不能（第 2 节走降级演示）
           __file__ = F:\ProGram\Python_Base\Agent\_py_source\07_protocols\01_acp智能体.py
```

三点说明：

1. **`deepagents` 与 `LocalShellBackend` 都是 `True`** —— 第 1 节的
   `create_deep_agent(...)` 能真的跑起来；本机缺的只有 `acp` / `deepagents-acp`；
2. **`能否真起 ACP stdio 服务端：不能`** —— 这就是第 2 节会走「降级演示」的原因。
   装了包以后这行会变成「能」，第 2.6 节的 `main()` 随之**自动切到真流程**，
   一行代码都不用改；
3. `__file__` 那一行是本 notebook 独有的适配：notebook 自己没有 `__file__`，
   我们把它指到归档的 ACP 服务端脚本，好让第 2.2 节生成的 `acp.json`
   **复制出去就能直接用**。

## 1. 课案原版：最短实现（46 行）

课案原版只做一件事：**造一个 Deep Agent，并保证「它作为一个模块被加载时」就已经存在**。

关键在这一行注释里：

> DeepAgents 的 ACP 适配器要求**模块暴露一个 `agent` 对象**（编译后的图）

也就是说 `01_acp智能体.py` 不是一个「跑起来做事的脚本」，而是一个**被拿去加载的模块**：
编辑器的 `acp.json` 里写「用哪个解释器跑哪个 .py」，`deepagents-acp` 把这个 .py
import 进来，从里面取走 `agent` 对象，然后挂到 stdio 上。

下面这一格**逐字复刻**课案原版（只删了文件头 docstring 与
`sys.stdout.reconfigure` 两处样板，见规范第 6 节）：

- `init_chat_model(model_provider="openai", ...)`：用「OpenAI 兼容」这一档去接
  本项目 `.env` 里的模型网关（`model_name` / `api_key` / `base_url` 全部来自 `settings`，
  一行都不硬编码）；
- `create_deep_agent(model=llm, tools=[], system_prompt=(...))`：连工具都不给，
  纯聊天 —— 先把「ACP 插座」这件事讲清楚，工具/子代理/技能是后面几课的事。

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "你是编码助手。可以读写文件、规划任务；"
        "回答尽量给出可直接执行的修改步骤。"
    ),
)

print("本文件由 deepagents-acp 加载运行（不要直接执行）")
print("启动命令：deepagents-acp --deepagents 07_protocols/acp智能体.py")

### 预期输出

```text
本文件由 deepagents-acp 加载运行（不要直接执行）
启动命令：deepagents-acp --deepagents 07_protocols/acp智能体.py
```

**注意这个反直觉的地方**：这一格看起来「什么都没干」—— 没有 `.invoke()`、没有输出回复。
它只是**把 agent 造出来放在模块命名空间里**，等别人来取。原脚本真正的「运行方式」是：

```text
编辑器 / deepagents-acp  →  import 这个模块  →  拿走 agent  →  挂到 stdio
```

所以原文件末尾的 `print` 才会写「不要直接执行」—— 直接执行它，只是白白加载了一堆东西
又什么都不发生（本机实测就是上面那两行）。

## 2. 完整版：把 Deep Agent 暴露成 ACP 服务端（stdio 模式）

原版 46 行把 agent 摆在那里，但**「怎么挂上插座」这一层被藏掉了**。
完整版把它补全，本节的三个知识点与课案 h4 小标题一一对应：

| # | 知识点 | 关键代码 |
|---|---|---|
| 1 | 安装 | `uv add deepagents-acp`（本项目用 uv；会连带装上 `acp`） |
| 2 | 快速上手 | `AgentServerACP(agent)` + `await run_agent(server)` —— 默认走 **stdio** |
| 3 | PyCharm 配置 | `~/.jetbrains/acp.json` 注册「用哪个解释器、跑哪个脚本」 |

先看课案原文的 `test.py`（27 行），它是整节的骨架。

### 2.1 课案原文：`test.py`（27 行）

课案到本项目的两处改写（规范第 3 节）：

1. `from conf import settings` → `from config import settings`；
2. `ChatOpenAI(model=..., api_key=..., base_url=...)` 这种显式三参数写法，
   本节是课案原文明确演示的形态，**允许保留**；但参数必须来自 `settings`，
   一个都不许硬编码。

下面把它存成字符串常量，等下由 `main()` 逐行打印出来对照 —— 这样读者不用回去翻
课案 HTML 就能看着原文读后面的真实流程。

In [ ]:
COURSE_TEST_PY = '''# test.py
import asyncio
from acp import run_agent
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents_acp.server import AgentServerACP
from langchain_openai import ChatOpenAI
from config import settings          # 课案原文是 from conf import settings


async def main() -> None:
    model = ChatOpenAI(
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )
    agent = create_deep_agent(
        model=model,
        system_prompt="你是一个编程助手",
        checkpointer=MemorySaver(),
        # 可自定义：tools、subagents、middleware、skills 等
        backend=LocalShellBackend(root_dir=".", virtual_mode=True, inherit_env=True),
    )

    server = AgentServerACP(agent)
    await run_agent(server)      # 默认 stdio：stdin 读取 → stdout 写回


if __name__ == "__main__":
    asyncio.run(main())
'''

### 为什么这一格没有输出

本格只做了一次字符串赋值，**没有输出**。它打印出来的样子见第 2.6 小节
`main()` 的输出（`main()` 的第一段就是逐行打印这段课案原文）。

### 2.2 PyCharm / JetBrains 注册：`acp.json`

编辑器要「知道去哪儿找你」，靠的是一份 JSON 配置。课案给的是 PyCharm / JetBrains 的
`C:\Users\<用户名>\.jetbrains\acp.json`：

| 字段 | 说明 |
|---|---|
| `command` | Python 解释器的**绝对路径**，Windows 下必须带 `.exe` 后缀 |
| `args` | 启动参数，这里传入你的 ACP 服务端脚本路径 |

课案步骤：

1. 创建/编辑 `C:\Users\<用户名>\.jetbrains\acp.json`；
2. 重启 PyCharm，打开 AI Chat，在 Agent 下拉菜单里会看到新增的 **"My Deep Agent"**
   （带自定义 agent 图标），选中即可使用；
3. PyCharm 会通过 **stdio 启动你的 Python 脚本**。

> ⚠️ **照抄课案会踩的坑**：
>
> - 课案里写的是 `C:\Users\13261\anaconda3\python.exe`，那是**课案作者的机器**。
>   本机必须填本仓 venv 的解释器（`F:\ProGram\Python_Base\.venv\Scripts\python.exe`）
>   —— **必须用装了 `deepagents-acp` 的那个解释器**，用错解释器 PyCharm 会以
>   `ModuleNotFoundError: No module named 'acp'` **静默退出**（界面上什么都不显示，
>   最难查的就是这种）。
> - JSON 里反斜杠是转义字符，Windows 路径**必须写成双反斜杠**。

下一格把课案那份（作者机器路径）和**本机可用版**（路径动态生成）都摆出来。
本机可用版用 `sys.executable`（= 正在跑本 notebook 的解释器，也就是 venv 那个）
和 `__file__`（第 0.1 小节已指向归档的 ACP 服务端脚本），所以复制出去就能用。

In [ ]:
COURSE_ACP_JSON = r'''{
    "agent_servers": {
        "My Deep Agent": {
            "command": "C:\\Users\\13261\\anaconda3\\python.exe",
            "args": ["C:\\Users\\13261\\Documents\\project\\ai_llm_jiaoan\\test.py"]
        }
    }
}'''


def build_local_acp_json() -> str:
    """把课案那份 acp.json 改写成「本机可直接用」的版本。

    课案用固定路径演示，这里用 sys.executable / __file__ 动态生成，
    免得学员复制过去还要手改路径。注意 Windows 路径在 JSON 里是 `\\`。
    """
    payload = {
        "agent_servers": {
            "My Deep Agent": {
                # sys.executable = 当前正在跑本文件的解释器，也就是 venv 的 python.exe
                "command": sys.executable,
                # __file__ = ACP 服务端脚本的绝对路径（第 0.1 小节已指向归档原版）
                "args": [str(__file__)],
            }
        }
    }
    # ensure_ascii=False：让中文（"My Deep Agent" 的键名可自定义）原样输出
    return json.dumps(payload, ensure_ascii=False, indent=4)

### 为什么这一格没有输出

同样是「只定义、不调用」，本格**没有输出**。两份 JSON 的真实打印结果
见第 2.6 小节 `main()` 输出的第二段。

顺带记住这句结论：**同一个 `test.py`，注册进哪个编辑器就能在哪个编辑器里用** ——
这正是「协议」的价值：适配一次，处处可用。

### 2.3 真流程：造 Agent → 挂到 stdio

这一格的 `build_agent()` + `serve_acp()` 就是课案 `test.py` 里 `main()` 的真身，
拆成两个函数是为了让职责清楚：

- `build_agent()`：组装 Deep Agent（模型 / 提示词 / 检查点 / 后端）；
- `serve_acp()`：把它交给 `AgentServerACP`，再用 `run_agent(server)` 挂到 **stdio** 上。

与课案相比的两点差异，都是为了「本机没装 `acp` 时也能看出结构」：

1. `model` 直接用课案的 `ChatOpenAI` + `settings` 三件套；
2. `backend` **只在 `LocalShellBackend` 导入成功时才传**，否则
   `create_deep_agent` 用默认 backend（不然这里就是 `NameError`）。

`LocalShellBackend` 的两个参数各有讲究：

| 参数 | 含义 | 不加会怎样 |
|---|---|---|
| `virtual_mode=True` | 文件操作被限制在 `root_dir` 下，路径穿越会被拦 | Agent 能读到你整台机器的任何文件 |
| `inherit_env=True` | 执行 shell 命令时**继承当前 Python 进程的所有环境变量** | 子进程 `PATH` / 代理 / 密钥全是空的，装依赖、跑构建都会失败 |

`MemorySaver()` 则把会话检查点放在内存里，**进程退出即丢**；生产环境应换成
`PostgresSaver`（见 `Agent/01_langgraph/03_短期记忆_生产.py`）。

> ⚠️ `serve_acp()` 一旦真的跑起来，**stdout 就被协议独占了**：
> 你自己 `print` 的任何东西都会污染协议流，调试信息只能走 `stderr`。
> 第 3.7 小节的子进程演示会把这条规则真的踩给你看。

In [ ]:
def build_agent():
    """按课案 test.py 组装 Deep Agent。"""
    from langchain_openai import ChatOpenAI
    from langgraph.checkpoint.memory import MemorySaver

    model = ChatOpenAI(
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )

    kwargs = {
        "model": model,
        "system_prompt": "你是一个编程助手",
        # MemorySaver：会话检查点放在内存里，进程退出即丢（生产换 PostgresSaver）
        "checkpointer": MemorySaver(),
        # 可自定义：tools、subagents、middleware、skills 等
    }
    if _HAS_LOCAL_SHELL_BACKEND:
        kwargs["backend"] = LocalShellBackend(
            root_dir=".", virtual_mode=True, inherit_env=True
        )
    return create_deep_agent(**kwargs)


async def serve_acp() -> None:
    """课案 test.py 的真身：把 Agent 交给 AgentServerACP，挂到 stdio 上。"""
    server = AgentServerACP(build_agent())
    # run_agent 默认走 stdio：从 stdin 读 JSON-RPC 请求，往 stdout 写响应/通知
    await run_agent(server)

### 2.4 降级演示：缺包时也要让学员看清「ACP 长什么样」

这一格是本课最实用的设计。`print_install_hint()` 把
「缺的是哪个模块名」原样打出来（学员能直接对着装），
`print_protocol_demo()` 则**手工构造一段真实格式的 ACP 会话报文**：

| 序 | 报文 | 有 id 吗 | 说明 |
|---|---|---|---|
| ① | `initialize` | 有（1） | 握手，交换协议版本与能力 |
| ① | `result` | 有（1） | 应答必须回同 id |
| ② | `session/new` | 有（2） | 建会话，服务端返回 `sessionId` |
| ② | `result` | 有（2） | 带回 `sess_0001` |
| ③ | `session/prompt` | 有（3） | 发用户消息（内容块数组，可夹带文件） |
| ④ | `session/update` | **无** | 通知：流式推回复的第一块 |
| ④ | `session/update` | **无** | 通知：第二块 |
| ③ | `result` | 有（3） | **最后才回**，带 `stopReason` |

三条要背下来的规则（第 3 节会**逐条用真代码验证**）：

1. **请求 / 响应成对**，靠 `id` 配对；
2. **通知没有 `id`**，服务端不回复（`session/update` 就是通知）；
3. **一行一个完整 JSON 对象，用 `\n` 分隔** —— 所以叫「行分隔 JSON-RPC」。

下面第二段代码里那句
`print("    # 或：... -m pip install deepagents-acp")` 是**本课唯一一处
为了适配 notebook 而改写的代码行**：源脚本里它是写死的绝对路径，而
`nbtool.py check` 明确禁止 code cell 里出现仓库绝对路径（会写坏别人的机器），
所以改成 `sys.executable`——效果更好（它就是当前解释器）且可移植。

In [ ]:
def print_install_hint() -> None:
    print("【前置条件检查】本机 venv 缺少 ACP 相关依赖：")
    for err in _IMPORT_ERRORS:
        # 把原始 ImportError 文本打出来，学生能直接看到「缺的是哪个模块名」
        print(f"    - {err}")
    print()
    print("请任选一种方式安装（本项目统一用 uv，不往全局环境装）：")
    print("    uv add deepagents-acp        # 会连带装上 acp（deepagents-acp 的依赖）")
    print("    # 或：uv add acp deepagents-acp")
    print(f"    # 或：{sys.executable} -m pip install deepagents-acp")
    print()


def print_protocol_demo() -> None:
    """手工构造一段 ACP 报文，让学员在缺包时也能看到协议的样子。"""
    transcript = [
        # ---- ① initialize：整个会话的第一条消息，必须先握手 ----
        (
            "→ ① 握手（客户端 → 服务端）",
            {
                "jsonrpc": "2.0",      # 固定字符串 "2.0"，不是数字 2.0
                "id": 1,               # 有 id = 请求，服务端必须回一条同 id 的响应
                "method": "initialize",  # 方法名，ACP 一共只有 6 个核心方法
                "params": {
                    "protocolVersion": 1,
                    "clientCapabilities": {"fs": {"readTextFile": True, "writeTextFile": True}},
                },
            },
        ),
        # ---- ① 的应答：id 必须与请求一致，靠它把响应配回请求 ----
        (
            "← ① 握手应答（服务端 → 客户端）",
            {
                "jsonrpc": "2.0",
                "id": 1,               # 和上面那条请求的 id 一一对应，这就是配对规则
                "result": {            # 成功用 result；失败则是 error，二者只出现一个
                    "protocolVersion": 1,
                    "agentCapabilities": {
                        "loadSession": True,
                        "promptCapabilities": {"image": False, "embeddedContext": True},
                    },
                },
            },
        ),
        # ---- ② session/new：建一个会话，后续所有消息都要带 sessionId ----
        (
            "→ ② 新建会话",
            {
                "jsonrpc": "2.0",
                "id": 2,
                "method": "session/new",
                # os.getcwd() = 仓库根，语义与课案里那个绝对路径完全一致
                "params": {"cwd": os.getcwd(), "mcpServers": []},
            },
        ),
        # 服务端生成 sessionId 并返回；下一节的真实收发里这个 id 会一路带在 params 里
        ("← ② 会话 ID", {"jsonrpc": "2.0", "id": 2, "result": {"sessionId": "sess_0001"}}),
        # ---- ③ session/prompt：唯一会「触发 Agent 干活」的方法 ----
        (
            "→ ③ 发送用户消息（prompt 是内容块数组，可夹带文件）",
            {
                "jsonrpc": "2.0",
                "id": 3,
                "method": "session/prompt",
                "params": {
                    "sessionId": "sess_0001",
                    "prompt": [{"type": "text", "text": "帮我看看这个项目怎么跑起来"}],
                },
            },
        ),
        # ---- ④ session/update：**没有 id**，所以是通知（notification）----
        (
            "← ④ 流式推送（**通知：没有 id**，客户端不回复）",
            {
                "jsonrpc": "2.0",
                "method": "session/update",     # 注意：这条没有 "id" 字段
                "params": {
                    "sessionId": "sess_0001",
                    # sessionUpdate 是通知的「子类型」，决定客户端怎么渲染这一块
                    "update": {
                        "sessionUpdate": "agent_message_chunk",   # 回复正文的一个片段
                        "content": {"type": "text", "text": "先执行 uv sync 安装依赖，"},
                    },
                },
            },
        ),
        # 同一次回复的第 2 个片段，客户端按到达顺序拼成完整回答
        (
            "← ④ 再来一块（同一次回复被切成很多块）",
            {
                "jsonrpc": "2.0",
                "method": "session/update",
                "params": {
                    "sessionId": "sess_0001",
                    "update": {
                        "sessionUpdate": "agent_message_chunk",
                        "content": {"type": "text", "text": "然后 uv run main.py。"},
                    },
                },
            },
        ),
        # ---- 最后才回 ③ 那条请求的响应（id=3）----
        (
            "← ③ 最后一个带 id 的响应：这一轮结束 + 结束原因",
            {"jsonrpc": "2.0", "id": 3, "result": {"stopReason": "end_turn"}},
        ),
    ]

    print("【降级演示】手工构造的 ACP 会话报文（真实协议格式，仅内容为示意）：")
    for title, message in transcript:
        # separators 去空格：输出更接近真实线路上的样子（一行一个 JSON 对象）
        print(f"    {title}")
        print("        " + json.dumps(message, ensure_ascii=False, separators=(",", ":")))
    print()
    print("记住三条规则（下一节会逐条验证）：")
    print("    ① 请求/响应成对，靠 id 配对；")
    print("    ② 通知没有 id，服务端不回复（session/update 就是通知）；")
    print("    ③ 一行一个完整 JSON 对象，用 \\n 分隔 —— 所以是「行分隔 JSON-RPC」。")
    print()

### 2.5 模型连通性自检

规范要求「留验证证据」：不管走真流程还是降级，都要证明 `settings` 三件套是通的。

这里刻意**不用 DeepAgent 去调**：DeepAgent 带 shell / file 工具，会真的在磁盘上
执行命令，作为「跑一遍看输出」的验收动作太重；直接问一句话就足以证明
`api_key` / `base_url` / `model_name` 通。

注意 `try/except Exception` —— 网络或额度问题不该让演示崩掉，只打印一行提示。

In [ ]:
def probe_model() -> None:
    """真调一次大模型，证明 settings 三件套可用。"""
    print("【模型连通性自检】settings.model_name =", settings.model_name)
    if not settings.api_key:
        print("    settings.api_key 为空，跳过自检（请检查根目录 .env）")
        return
    try:
        llm = init_chat_model(
            model_provider="openai",
            model=settings.model_name,
            api_key=settings.api_key,
            base_url=settings.base_url,
        )
        reply = llm.invoke("只回复两个字：可用")
        text = reply.content if isinstance(reply.content, str) else str(reply.content)
        print("    模型回复：", text.replace("\n", " ")[:60])
    except Exception as exc:  # 网络/额度问题不该让演示崩掉
        print(f"    模型调用失败（{type(exc).__name__}）：{exc}")

### 2.6 主流程 `main()`：把上面全部串起来

源文件里对 `main()` 的定位写得很明确：

> 整个 `main` 就是「教学设计」本身：先把课案原文摊开给学员看，
> 再看本机有没有装依赖 —— 装了走真实链路，没装就用手工报文演示协议长什么样。

四步走：

| 步 | 做什么 |
|---|---|
| 1 | 逐行打印课案 `test.py` 原文（不用回去翻 HTML） |
| 2 | 打印课案那份 `acp.json` + 本机可用版 |
| 3 | 依赖检查：**齐了真的起 stdio 服务端**（会阻塞），**缺了走降级演示** |
| 4 | 无论走哪条，都做一次模型连通性自检 |

本机是「缺包」分支，所以下面这一格**不会阻塞**，跑完就结束。
装包之后它会真的起服务 —— 那种情况下这一格要手动中断，见「运行条件」里的提醒。

In [ ]:
def main() -> None:
    print("=" * 66)
    print("ACP 实操：把 Deep Agent 暴露为 ACP 服务端（stdio 模式）")
    print("=" * 66)
    print()

    # ---------- 第 1 步：看课案原文 ----------
    print("【课案原文】test.py（27 行，本项目已把 conf → config）：")
    for line in COURSE_TEST_PY.splitlines():
        print("    " + line)
    print()

    # ---------- 第 2 步：PyCharm / JetBrains 注册配置 ----------
    print("【课案原文】C:\\Users\\<用户名>\\.jetbrains\\acp.json：")
    for line in COURSE_ACP_JSON.splitlines():
        print("    " + line)
    print()
    print("【本机可用版】把上面的路径换成动态生成的（可直接复制）：")
    for line in build_local_acp_json().splitlines():
        print("    " + line)
    print()

    # ---------- 第 3 步：装没装？ ----------
    print_install_hint()
    if _ACP_READY:
        print("【依赖检查】acp + deepagents-acp + deepagents 均已就绪，启动真实 ACP 服务端。")
        print("            （注意：stdio 服务端会阻塞等待编辑器的 JSON-RPC 请求，Ctrl+C 退出）")
        print()
        try:
            asyncio.run(serve_acp())
        except KeyboardInterrupt:
            # 服务端本来就是长驻进程，手动中断属于正常退出路径，不算失败
            print("\n已手动中断 ACP 服务端。")
            return
        except Exception as exc:
            # 起不来时给出最可能的两个原因，而不是甩 traceback
            print(f"启动失败（{type(exc).__name__}）：{exc}")
            print("常见原因：解释器不是装 deepagents-acp 的那个；或编辑器未按协议发起 initialize。")
            return
    else:
        print("【降级演示】未安装 acp / deepagents-acp，无法真的 run_agent(server)。")
        print("            下面用「手工构造报文」的方式展示协议交互，安装后本文件会自动切到真流程。")
        print()
        print_protocol_demo()

    # ---------- 第 4 步：确认大模型侧配置没问题 ----------
    probe_model()
    print()
    # 小结把「协议解决什么问题」再收一句口，并指向下一节的真实收发
    print("=" * 66)
    print("小结：ACP = 编辑器与 Agent 之间的一层「LSP 式」标准协议。")
    print("      你只写 create_deep_agent(...) + run_agent(server)，")
    print("      报文封装、stdio 读写、会话管理都由 deepagents-acp 代劳。")
    print("      想知道它到底在传什么 → 继续看 02_acp原理_jxsd.py。")
    print("=" * 66)


main()

### 预期输出

本机实测（`deepagents` 已装、`acp` / `deepagents-acp` 缺 → 走降级分支）：

```text
==================================================================
ACP 实操：把 Deep Agent 暴露为 ACP 服务端（stdio 模式）
==================================================================

【课案原文】test.py（27 行，本项目已把 conf → config）：
    # test.py
    import asyncio
    from acp import run_agent
    from deepagents import create_deep_agent
    from langgraph.checkpoint.memory import MemorySaver
    from deepagents_acp.server import AgentServerACP
    from langchain_openai import ChatOpenAI
    from config import settings          # 课案原文是 from conf import settings
    
    
    async def main() -> None:
        model = ChatOpenAI(
            model=settings.model_name,
            api_key=settings.api_key,
            base_url=settings.base_url,
        )
        agent = create_deep_agent(
            model=model,
            system_prompt="你是一个编程助手",
            checkpointer=MemorySaver(),
            # 可自定义：tools、subagents、middleware、skills 等
            backend=LocalShellBackend(root_dir=".", virtual_mode=True, inherit_env=True),
        )
    
        server = AgentServerACP(agent)
        await run_agent(server)      # 默认 stdio：stdin 读取 → stdout 写回
    
    
    if __name__ == "__main__":
        asyncio.run(main())

【课案原文】C:\Users\<用户名>\.jetbrains\acp.json：
    {
        "agent_servers": {
            "My Deep Agent": {
                "command": "C:\\Users\\13261\\anaconda3\\python.exe",
                "args": ["C:\\Users\\13261\\Documents\\project\\ai_llm_jiaoan\\test.py"]
            }
        }
    }

【本机可用版】把上面的路径换成动态生成的（可直接复制）：
    {
        "agent_servers": {
            "My Deep Agent": {
                "command": "F:\\ProGram\\Python_Base\\.venv\\Scripts\\python.exe",
                "args": [
                    "F:\\ProGram\\Python_Base\\Agent\\_py_source\\07_protocols\\01_acp智能体.py"
                ]
            }
        }
    }

【前置条件检查】本机 venv 缺少 ACP 相关依赖：
    - acp：No module named 'acp'
    - deepagents-acp：No module named 'deepagents_acp'

请任选一种方式安装（本项目统一用 uv，不往全局环境装）：
    uv add deepagents-acp        # 会连带装上 acp（deepagents-acp 的依赖）
    # 或：uv add acp deepagents-acp
    # 或：F:\ProGram\Python_Base\.venv\Scripts\python.exe -m pip install deepagents-acp

【降级演示】未安装 acp / deepagents-acp，无法真的 run_agent(server)。
            下面用「手工构造报文」的方式展示协议交互，安装后本文件会自动切到真流程。

【降级演示】手工构造的 ACP 会话报文（真实协议格式，仅内容为示意）：
    → ① 握手（客户端 → 服务端）
        {"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":1,"clientCapabilities":{"fs":{"readTextFile":true,"writeTextFile":true}}}}
    ← ① 握手应答（服务端 → 客户端）
        {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":1,"agentCapabilities":{"loadSession":true,"promptCapabilities":{"image":false,"embeddedContext":true}}}}
    → ② 新建会话
        {"jsonrpc":"2.0","id":2,"method":"session/new","params":{"cwd":"F:\\ProGram\\Python_Base","mcpServers":[]}}
    ← ② 会话 ID
        {"jsonrpc":"2.0","id":2,"result":{"sessionId":"sess_0001"}}
    → ③ 发送用户消息（prompt 是内容块数组，可夹带文件）
        {"jsonrpc":"2.0","id":3,"method":"session/prompt","params":{"sessionId":"sess_0001","prompt":[{"type":"text","text":"帮我看看这个项目怎么跑起来"}]}}
    ← ④ 流式推送（**通知：没有 id**，客户端不回复）
        {"jsonrpc":"2.0","method":"session/update","params":{"sessionId":"sess_0001","update":{"sessionUpdate":"agent_message_chunk","content":{"type":"text","text":"先执行 uv sync 安装依赖，"}}}}
    ← ④ 再来一块（同一次回复被切成很多块）
        {"jsonrpc":"2.0","method":"session/update","params":{"sessionId":"sess_0001","update":{"sessionUpdate":"agent_message_chunk","content":{"type":"text","text":"然后 uv run main.py。"}}}}
    ← ③ 最后一个带 id 的响应：这一轮结束 + 结束原因
        {"jsonrpc":"2.0","id":3,"result":{"stopReason":"end_turn"}}

记住三条规则（下一节会逐条验证）：
    ① 请求/响应成对，靠 id 配对；
    ② 通知没有 id，服务端不回复（session/update 就是通知）；
    ③ 一行一个完整 JSON 对象，用 \n 分隔 —— 所以是「行分隔 JSON-RPC」。

【模型连通性自检】settings.model_name = deepseek-flash
    模型回复： 可用

==================================================================
小结：ACP = 编辑器与 Agent 之间的一层「LSP 式」标准协议。
      你只写 create_deep_agent(...) + run_agent(server)，
      报文封装、stdio 读写、会话管理都由 deepagents-acp 代劳。
      想知道它到底在传什么 → 继续看 02_acp原理_jxsd.py。
==================================================================
```

三点提醒：

1. `settings.model_name` 与「模型回复」两行**随你的 `.env` 和模型当时的心情而变**
   （本机实测是 `deepseek-flash` / `可用`）；
2. `【本机可用版】` 里的两条路径来自 `sys.executable` 和 `__file__`，
   在你自己机器上会是你的路径 —— 这正是它「可直接复制」的原因；
3. `【课案原文】` 里那两处 `C:\Users\13261\...` **不是本机路径**，是课案作者的机器，
   保留原样是为了让你看清「照抄会踩什么坑」。

### 输出导读：这一大段到底在说什么

| 段落 | 在讲什么 | 你该记住的 |
|---|---|---|
| `【课案原文】test.py` | 完整版骨架（27 行） | `AgentServerACP(agent)` + `await run_agent(server)` 就是「插上 ACP 插座」的全部 |
| `【课案原文】acp.json` | 编辑器侧的注册文件 | 两个字段：`command`（解释器绝对路径）+ `args`（脚本路径） |
| `【本机可用版】` | 动态生成、复制即用 | 用错解释器是**静默失败**，必须指向装了 `deepagents-acp` 的那个 |
| `【前置条件检查】` | 缺哪两个包、怎么装 | 本项目统一 `uv add`，不往全局环境装 |
| `【降级演示】手工构造报文` | 真实格式的 9 条报文 | 有 id = 请求/响应；**无 id = 通知**；一行一个 JSON |
| `【模型连通性自检】` | `settings` 三件套证明可用 | 🟡 档位的意义就在这一行 |

## 3. 原理：把 JSON-RPC 2.0 报文自己收一发

上一节 `run_agent(server)` 一行就把 Agent 挂上了 stdio，但「**底下到底在传什么**」
被 `deepagents-acp` 封装掉了。这一节把那层封装**撕开**：

1. 讲透 JSON-RPC 2.0 的报文形态（请求 / 响应 / 通知 / 批量，以及判定规则）；
2. 讲透 ACP 的核心方法；
3. **真的跑一遍** —— 而且是两种跑法：
   - **演示 ①：内存双向管道** —— 客户端协程 ↔ 服务端协程，走完整会话；
   - **演示 ②：真实子进程管道** —— `subprocess.Popen` 起一个 Python 子进程当 ACP 服务端，
     从它的 stdout 按行读**真的** JSON —— 这就是 PyCharm 启动你脚本时发生的事。

全程**只用标准库**（`asyncio` / `json` / `os` / `subprocess` / `sys`），
不依赖 `acp` 包，所以**本机没装 `acp` 也能看到协议的真实收发**。
这也是本 notebook 最「空手可跑」的一节。

### 3.1 JSON-RPC 2.0：四种消息形态

课案原文：JSON-RPC 2.0 是一种轻量级远程过程调用协议，用 JSON 编码请求和响应，
**与传输层无关**（可跑在 HTTP、stdio、WebSocket 之上）。

| 特征 | 说明 |
|---|---|
| 请求格式 | `{"jsonrpc": "2.0", "id": 1, "method": "方法名", "params": {...}}` |
| 响应格式 | `{"jsonrpc": "2.0", "id": 1, "result": {...}}` 或 `{"id":1,"error":{...}}` |
| 通知（Notification） | **不带 id 的请求**，服务端**不回复**（如 `session/update` 流式推送） |
| 批量请求 | 一次发多个请求对象组成的数组，服务端返回对应数组 |
| 传输无关 | 只定义消息格式，不限制传输方式 —— stdio、HTTP、WebSocket 均可 |

#### 四条铁律（背下来，A2A 那三节还会再见）

1. `jsonrpc` 字段固定是**字符串** `"2.0"`，不是数字 `2.0`；
2. **有 id 必有回**：请求一定收到一条同 id 的响应（`result` 或 `error` 二选一）；
3. **无 id 不回**：通知是单向的，服务端处理完就算；
4. id 由**发起方**分配，只在一次会话内唯一，用来把响应配回请求。

#### 拿到一条 JSON，怎么判断它属于哪一种

判据看的是**字段组合**，不是内容，所以规则可以穷举：

| 判定顺序 | 报文形态 | 判据（字段组合） | 收到方要做什么 |
|---|---|---|---|
| 第 1 步 | 批量请求/响应 | 顶层是 **JSON 数组** `[...]` | 逐条按下表判定，把「有响应的」收集成数组回；数组里可能混着请求和通知 |
| 第 2 步 | 请求 | 对象里**有 `method` 且有 `id`** | 处理它，然后回一条**同 id** 的响应 |
| 第 3 步 | 通知 | 对象里**有 `method` 但没有 `id`**（或 `id` 为 `null`） | 处理它，**不回任何东西**（回了就是协议错误） |
| 第 4 步 | 响应 | 对象里**没有 `method`，有 `id`**，并带 `result` 或 `error` | 按 id 找到等待中的请求，把 result/error 交给它 |

#### 两个边界情况（第 3.6 / 3.7 小节都真的构造出来跑过）

- **`id: null` 的错误响应** —— 请求连 JSON 都解析不出来（`-32700` Parse error）时，
  规范要求回一条 `"id": null` 的 error，因为此时根本不知道对方用的是哪个 id；
- **批量请求返回的数组可以比请求数组短** —— 数组里的通知不产生响应条目，
  所以「两个请求 + 一个通知」进去，出来只有两条响应。

还有一条不属于 JSON-RPC、但属于 ACP 传输层的规则：

> **行分隔**：一行 = 一个完整 JSON 对象，用 `\n` 分隔（不是长度前缀、不是大数组）。
> 所以读的一方可以按行解析、按行处理，天然支持流式。

#### JSON-RPC 标准错误码

| 码 | 含义 | 典型场景 |
|---|---|---|
| `-32700` | Parse error | 收到的根本不是合法 JSON |
| `-32600` | Invalid Request | 缺 `jsonrpc` / `method` 字段，结构不对 |
| `-32601` | Method not found | 方法名不认识（例如拼错 `session/new`） |
| `-32602` | Invalid params | 参数缺失或类型不对（例如没传 `cwd`） |
| `-32603` | Internal error | 服务端内部异常 |
| `-32000` ~ `-32099` | 服务端自定义错误码 | ACP 用它传业务错误（例如「会话不存在」`-32001`） |

下面第 3.6 小节的演示会把 `-32601` / `-32602` / `-32001` 全部真的触发一遍，
第 3.7 小节触发 `-32700`。

### 3.2 消息构造函数：一眼看清四种形态的字段差异

四个小函数，把「请求 / 通知 / 成功响应 / 失败响应」的字段差异钉死。
后面所有演示都用它们造报文，不手写字典 —— 这样字段拼错会立刻暴露。

注意 `rpc_error` 的 `req_id` 类型是 `int | None`：允许 `None`，因为
Parse error 时根本不知道对方用的哪个 id。

In [ ]:
def rpc_request(method: str, params: dict, req_id: int) -> dict:
    """请求：有 id，等对方回。"""
    return {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params}


def rpc_notification(method: str, params: dict) -> dict:
    """通知：**不带 id**，对方不回复。ACP 的 session/update / session/cancel 都是它。"""
    return {"jsonrpc": "2.0", "method": method, "params": params}


def rpc_result(req_id: int, result: dict) -> dict:
    """成功响应：id + result，两个字段就够了。"""
    return {"jsonrpc": "2.0", "id": req_id, "result": result}


def rpc_error(req_id: int | None, code: int, message: str, data=None) -> dict:
    """失败响应：id + error{code,message,data}，**不能同时有 result**。

    id 可以是 None —— 当请求的 id 都解析不出来时（例如 JSON 坏了），
    规范要求回一条 `"id": null` 的错误。
    """
    err = {"jsonrpc": "2.0", "id": req_id, "error": {"code": code, "message": message}}
    if data is not None:
        err["error"]["data"] = data
    return err

### 3.3 ACP 的核心方法，与一次会话的报文往返

课案原文：ACP 定义的核心方法只有 6 个，其中 5 个是请求（客户端 → 服务端），
1 个是通知（服务端 → 客户端）：

| 方法 | 一句话 |
|---|---|
| `initialize` | 握手，交换协议版本和能力 |
| `session/new` | 新建会话，返回 `sessionId` |
| `session/load` | 恢复已有会话 |
| `session/prompt` | 发送用户消息，Agent 处理后返回 `stopReason` |
| `session/cancel` | 取消当前会话中的执行（**通知**） |
| `session/update` | **通知**：Agent 流式推送状态，客户端不要求回复 |

`session/update` 靠 `sessionUpdate` **子类型**区分推的是什么：

| 子类型 | 内容 |
|---|---|
| `agent_message_chunk` | 回复正文的一个片段 |
| `agent_thought_chunk` | 思考链片段 |
| `tool_call` | 发起一次工具调用 |
| `tool_call_update` | 工具调用状态更新 |
| `plan` | 执行计划 |

**方向补充**（课案表格没写全，但实际会用到）：

- 客户端 → 服务端：`initialize` / `session/new` / `session/load` /
  `session/prompt` / `session/cancel`；
- 服务端 → 客户端：`session/update`（通知），以及 `fs/read_text_file`、
  `fs/write_text_file`、`session/request_permission` 这几个**反向请求**
  —— 因为「读文件」「要用户批准执行命令」这两件事，光靠 Agent 自己做不到。

#### 报文往返图（sequenceDiagram）

```mermaid
sequenceDiagram
    autonumber
    participant C as 编辑器（ACP 客户端）
    participant A as 你的 Agent 进程（ACP 服务端，stdio）
    C->>A: initialize（有 id=1，请求）
    A-->>C: result（id=1，响应）
    C->>A: session/new（有 id=2）
    A-->>C: result {"sessionId":"sess_0001"}（id=2）
    C->>A: session/prompt（有 id=3）
    A-->>C: session/update（**无 id**，通知）
    A-->>C: session/update（**无 id**，通知 × N）
    A-->>C: result {"stopReason":"end_turn"}（id=3，最后才回）
```

**等价文本（裸 JupyterLab 不渲染 mermaid，看这里）**：

```text
→ {"jsonrpc":"2.0","id":1,"method":"initialize","params":{...}}     有 id = 请求，必须回一条同 id 的响应
← {"jsonrpc":"2.0","id":1,"result":{...}}
→ {"jsonrpc":"2.0","id":2,"method":"session/new","params":{"cwd":"...","mcpServers":[]}}
← {"jsonrpc":"2.0","id":2,"result":{"sessionId":"sess_0001"}}
→ {"jsonrpc":"2.0","id":3,"method":"session/prompt","params":{"sessionId":"sess_0001","prompt":[...]}}
← {"jsonrpc":"2.0","method":"session/update","params":{...}}         无 id = 通知，客户端不用回（流式推送）
← {"jsonrpc":"2.0","method":"session/update","params":{...}}         可以推任意多条
← {"jsonrpc":"2.0","id":3,"result":{"stopReason":"end_turn"}}        顺序关键：通知推完，最后才回响应
```

**三个必须记住的顺序细节**：

1. 一次 `session/prompt` 期间，服务端可以按自己的节奏推**任意多条** `session/update`；
2. **所有通知推完，最后才回带 id 的那条响应** —— 顺序反了，客户端就会提前结束本轮；
3. 响应里的 `stopReason` 说明这一轮为什么结束：`end_turn`（正常说完）/
   `max_tokens` / `refusal` / `cancelled`（被取消）。

传输层细节（课案原文）：客户端（PyCharm）通过 **stdin** 发 JSON-RPC 请求，
服务端（你的 Python 脚本）处理后从 **stdout** 回复；
**每一行是一个完整的 JSON 对象，用 `\n` 分隔**。
`AgentServerACP` 和 `run_agent()` 已经把这些全封装好了，你只需关注 agent 的业务逻辑。

### 3.4 假的 ACP 服务端：纯逻辑，不依赖任何第三方包

真服务端里，这里应该是 `AgentServerACP(deep_agent)`。为了**不调模型、不联网**，
我们用一个「固定回复 + 分块吐出」的假 Agent，只演示协议层。

`FakeAcpServer` 同时扮演两个角色 —— ACP 的现实就是这样，**双向都对开**：

- **服务端**：处理 `initialize` / `session/new` / `session/prompt` / `session/load`；
- **客户端**：主动向对端发 `session/update` 通知（流式推送）。

它实现了四件事：

| 方法 | 作用 |
|---|---|
| `handle()` | 入口：是数组就按**批量**逐条处理，把有结果的收集成数组回 |
| `_dispatch()` | 单条分发：结构校验 → 按 `method` 分支 → 返回响应或 `None`（通知） |
| `_stream_message()` | 把回复切成若干 `agent_message_chunk` 通知推出去 |
| `run_forever()` | 常驻循环：从 `inbound` 队列取一条处理一条 |

三个值得停一下看的实现细节：

1. **`self._next_call = 100`**：服务端反向请求用的 id 段，**和客户端的 id 错开**，
   避免双向同时发请求时 id 撞车（真实 ACP 服务端也会这么分区间）；
2. **参数校验失败回 `-32602` 而不是抛异常** —— 抛异常会把整条连接搞崩，
   而「参数不对」本来就在协议的错误码表里；
3. **`session/prompt` 的顺序**：先推 `agent_thought_chunk`（思考链），
   再推一串 `agent_message_chunk`（正文），**最后**才 `return rpc_result(...)`。

In [ ]:
FAKE_AGENT_REPLY = "先执行 uv sync 安装依赖，然后 uv run main.py 启动。"


class FakeAcpServer:
    """内存版 ACP 服务端：收请求 → 分发 → 回响应，顺手推 session/update 通知。"""

    def __init__(self, inbound: "asyncio.Queue[dict]", outbound: "asyncio.Queue[dict]") -> None:
        self.inbound = inbound  # 对端 → 我
        self.outbound = outbound  # 我 → 对端
        self.sessions: dict[str, dict] = {}
        self._next_session = 0
        self._next_call = 100  # 服务端反向请求用的 id 段，和客户端 id 错开，避免撞车

    # ---------- 4.1 发送侧 ----------
    async def send(self, message: dict) -> None:
        await self.outbound.put(message)

    async def notify(self, method: str, params: dict) -> None:
        """推一条通知出去（无 id）。"""
        await self.send(rpc_notification(method, params))

    async def _stream_message(self, session_id: str, text: str, chunk_size: int = 12) -> None:
        """把回复切成若干 agent_message_chunk 通知推出去 —— 这就是协议层的流式输出。"""
        for i in range(0, len(text), chunk_size):
            # 用「无 id 的通知」一句一句推：发送方不等回复，接收方收到一块渲染一块
            await self.notify(
                "session/update",
                {
                    "sessionId": session_id,
                    "update": {
                        "sessionUpdate": "agent_message_chunk",
                        "content": {"type": "text", "text": text[i : i + chunk_size]},
                    },
                },
            )

    # ---------- 4.2 处理侧 ----------
    async def handle(self, message) -> None:
        """收到一条消息（可能是请求、通知，或批量数组）。"""
        if isinstance(message, list):
            # 批量：逐条处理，把「有结果的」收集成数组一次性回；通知不产生响应条目
            responses = []
            for item in message:
                resp = await self._dispatch(item)
                if resp is not None:
                    responses.append(resp)
            if responses:
                await self.send(responses)
            return

        resp = await self._dispatch(message)
        if resp is not None:
            await self.send(resp)

    async def _dispatch(self, message: dict) -> dict | None:
        """单条分发。返回 None 表示「不回复」（通知，或通知类方法）。"""
        # --- 结构校验：缺 jsonrpc / method 一律 -32600 ---
        if not isinstance(message, dict) or message.get("jsonrpc") != "2.0":
            return rpc_error(None, -32600, "Invalid Request：缺少 jsonrpc 2.0 字段")

        method = message.get("method")
        params = message.get("params") or {}
        req_id = message.get("id")
        is_notification = req_id is None  # 没 id 就是通知，处理完不回

        if method == "initialize":
            # 握手：双方按「都支持的能力」往下走（客户端不支持写文件，
            # 服务端就不该发 fs/write_text_file 反向请求）
            client_caps = params.get("clientCapabilities", {})
            return rpc_result(
                req_id,
                {
                    "protocolVersion": min(params.get("protocolVersion", 1), 1),
                    "agentCapabilities": {
                        "loadSession": True,
                        "promptCapabilities": {
                            # 客户端连读文件都不会，就别指望它提供上下文
                            "embeddedContext": bool(client_caps.get("fs", {}).get("readTextFile")),
                            "image": False,
                        },
                    },
                },
            )

        if method == "session/new":
            cwd = params.get("cwd")
            if not cwd:
                # 参数校验失败 → -32602，而不是抛异常把连接搞崩
                return rpc_error(req_id, -32602, "Invalid params：session/new 必须带 cwd")
            self._next_session += 1
            session_id = f"sess_{self._next_session:04d}"
            # 建会话时就把 cwd / mcpServers 记下来，后面 session/prompt 只带 sessionId 就能找回上下文
            self.sessions[session_id] = {"cwd": cwd, "mcpServers": params.get("mcpServers", [])}
            return rpc_result(req_id, {"sessionId": session_id})

        if method == "session/load":
            session_id = params.get("sessionId")
            if session_id not in self.sessions:
                # 业务级错误用服务端自定义码（-32000 ~ -32099）
                return rpc_error(req_id, -32001, "会话不存在", {"sessionId": session_id})
            return rpc_result(req_id, {"sessionId": session_id})

        if method == "session/prompt":
            session_id = params.get("sessionId")
            if session_id not in self.sessions:
                return rpc_error(req_id, -32001, "会话不存在", {"sessionId": session_id})
            # ⚠️ 顺序关键：先推完所有 session/update，最后才回这条带 id 的响应
            # 先推一条「思考中」再推正文 —— 真实 ACP 的常见形态
            await self.notify(
                "session/update",
                {
                    "sessionId": session_id,
                    "update": {
                        "sessionUpdate": "agent_thought_chunk",
                        "content": {"type": "text", "text": "（思考）学员问的是启动步骤…"},
                    },
                },
            )
            await self._stream_message(session_id, FAKE_AGENT_REPLY)
            return rpc_result(req_id, {"stopReason": "end_turn"})

        if method == "session/cancel":
            # 通知：不回。真实服务端收到后要把进行中的 prompt 以
            # stopReason="cancelled" 收尾，并撤掉所有待批准的 request_permission
            print(f"        [服务端] 收到 session/cancel 通知（无 id），停止会话 {params.get('sessionId')}")
            return None

        return rpc_error(req_id, -32601, f"Method not found：{method}")

    async def run_forever(self) -> None:
        while True:
            message = await self.inbound.get()
            await self.handle(message)

### 3.5 假的 ACP 客户端：按 id 配对，收通知

`FakeAcpClient` 扮演 PyCharm。核心是 `pending` 这张表：**id → Future**。

| 方法 | 作用 |
|---|---|
| `call()` | 分配 id → 发请求 → `await` 那条**同 id** 的响应 |
| `read_loop()` | 常驻读循环：把响应配给等待中的 Future，把通知丢进 `notifications` |
| `_route()` | 单条路由：**有 id 且不为 None** → 找 Future；否则当通知收下 |

真实客户端读的是 **stdin 的每一行**；这里读的是队列，但
「一行一个 JSON 对象」的解帧逻辑完全一样 —— 这就是 `call()` 能「一边等响应、
一边把流式通知收进列表」的原因（也是第 3.6 节必须用 `asyncio` 的原因：
同步写法做不出「一边等响应一边收通知」）。

In [ ]:
class FakeAcpClient:
    """内存版 ACP 客户端（扮演 PyCharm）：发请求、按 id 收响应、收通知。"""

    def __init__(self, inbound: "asyncio.Queue[dict]", outbound: "asyncio.Queue[dict]") -> None:
        self.inbound = inbound  # 服务端 → 我
        self.outbound = outbound  # 我 → 服务端
        self.pending: "dict[int, asyncio.Future]" = {}
        self.notifications: "list[dict]" = []
        self._req_id = 0
        self._reader: "asyncio.Task | None" = None

    def _new_id(self) -> int:
        self._req_id += 1
        return self._req_id

    async def call(self, method: str, params: dict) -> dict:
        """发一个请求并**等它那条同 id 的响应**（通知会被顺路收集起来）。"""
        req_id = self._new_id()
        fut: "asyncio.Future" = asyncio.get_running_loop().create_future()
        self.pending[req_id] = fut
        await self.outbound.put(rpc_request(method, params, req_id))
        return await fut

    async def notify(self, method: str, params: dict) -> None:
        await self.outbound.put(rpc_notification(method, params))

    async def read_loop(self) -> None:
        """常驻读循环：把响应配给等待的 Future，把通知丢进 notifications 列表。"""
        while True:
            message = await self.inbound.get()
            if isinstance(message, list):  # 批量响应
                for item in message:
                    self._route(item)
                continue
            self._route(message)

    def _route(self, message: dict) -> None:
        if "id" in message and message["id"] is not None:
            fut = self.pending.pop(message["id"], None)
            if fut and not fut.done():
                fut.set_result(message)
                return
        # 没有 id：通知
        self.notifications.append(message)

### 3.6 演示 ①：内存双向管道（离线可跑的核心演示）

两条 `asyncio.Queue` 当「两根管子」，服务端和客户端各拿一头：

```text
  客户端  --c2s-->  服务端
  客户端  <--s2c--  服务端
```

七步走，每一步都对应 JSON-RPC 的一条规则：

| 步 | 动作 | 验证的规则 |
|---|---|---|
| [1] | 批量请求（两条）| 数组进 → **数组出**，可混合成功与失败；`session/nwe` 拼错 → `-32601` |
| [2] | `initialize` | 有 id 必有回；双方交换能力 |
| [3] | `session/new` | 服务端分配 `sessionId` |
| [4] | 故意漏掉 `cwd` | 参数校验失败 → `-32602`（而不是崩连接） |
| [5] | `session/prompt` | **先收一串无 id 通知，最后才收到带 id 的响应** |
| [6] | 方法名拼错 | `-32601` Method not found |
| [7] | `session/cancel` | **通知不回复**，客户端收到 0 条新消息 |

> ⚠️ **notebook 专属改动（本课唯一的异步适配）**
>
> 源脚本末尾是 `asyncio.run(demo_memory_pipe())`。这在普通脚本里没问题，
> 但在 **notebook 内核里会直接抛**
> `RuntimeError: asyncio.run() cannot be called from a running event loop`
> （本机实测，见下面的 `run_coro` 与本课「常见坑」第 3 条）。
>
> 所以这里加了一个 5 行的 `run_coro()`：**脚本环境仍走 `asyncio.run`**，
> 检测到「已有运行中的事件循环」（就是 notebook 内核）时，
> 另起一条线程给它一个干净的新循环。业务代码 `demo_memory_pipe()` **一个字没改**。

第 [1] 步的批量请求是故意脱开读循环做的（直接手取队列），
免得和后面启动的 `read_loop` 抢队列 —— 注释里也写了。

In [ ]:
import threading


def run_coro(coro):
    """把协程跑完并返回结果 —— 普通脚本与 notebook 内核两种环境都适用。

    脚本里 `asyncio.get_running_loop()` 会抛 RuntimeError → 照源文件用 asyncio.run；
    notebook 内核里已经有一个正在运行的事件循环 → asyncio.run 会抛
    `RuntimeError: asyncio.run() cannot be called from a running event loop`（本机实测），
    所以另起一条线程，给协程一个干净的新循环。
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)          # ← 普通脚本路径（源文件原本的写法）

    box: dict = {}

    def _worker() -> None:
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as exc:      # noqa: BLE001 —— 把异常原样带回主线程
            box["error"] = exc

    worker = threading.Thread(target=_worker)
    worker.start()
    worker.join()
    if "error" in box:
        raise box["error"]
    return box.get("value")


async def demo_memory_pipe() -> None:
    """演示 ①：内存双向管道跑完整会话。"""
    print("=" * 66)
    print("演示 ① 内存管道：客户端协程 ↔ 服务端协程，完整 ACP 会话")
    print("=" * 66)

    c2s: "asyncio.Queue[dict]" = asyncio.Queue()  # 客户端 → 服务端
    s2c: "asyncio.Queue[dict]" = asyncio.Queue()  # 服务端 → 客户端
    server = FakeAcpServer(inbound=c2s, outbound=s2c)
    client = FakeAcpClient(inbound=s2c, outbound=c2s)

    server_task = asyncio.create_task(server.run_forever())

    def show(direction: str, message) -> None:
        """打印一行报文 —— 行分隔 JSON-RPC 的真实观感。"""
        text = json.dumps(message, ensure_ascii=False, separators=(",", ":"))
        print(f"    {direction} {text}")

    # ---------- ① 批量请求：先脱开读循环，直观看到「数组进、数组出」 ----------
    print("\n[1] 批量请求：一个数组里塞两条，服务端回同样结构的数组")
    print("    （JSON-RPC 规范：批量请求可以混合成功与失败，逐条独立结算）")
    batch = [
        rpc_request("initialize", {"protocolVersion": 1}, 101),
        rpc_request("session/nwe", {}, 102),  # 故意拼错方法名
    ]
    show("→", batch)
    await c2s.put(batch)
    # 读循环还没启动，直接手取 —— 省得和 read_loop 抢队列
    show("←", await s2c.get())
    print()

    # ---------- 启动常驻读循环：后面的请求/响应都靠它按 id 配对 ----------
    reader_task = asyncio.create_task(client.read_loop())

    # ---------- ② initialize：握手 ----------
    print("[2] 握手：initialize（交换协议版本与能力）")
    init_params = {
        "protocolVersion": 1,
        "clientCapabilities": {"fs": {"readTextFile": True, "writeTextFile": True}},
    }
    print("    → " + json.dumps(rpc_request("initialize", init_params, 1),
                               ensure_ascii=False, separators=(",", ":")))
    # client.call 会分配 id、发出请求、并 await 那条同 id 的响应
    resp = await client.call("initialize", init_params)
    show("←", resp)

    # ---------- ③ session/new：建会话 ----------
    print("\n[3] 建会话：session/new（必须给 cwd，否则 -32602）")
    new_params = {"cwd": os.getcwd(), "mcpServers": []}
    print("    → " + json.dumps(rpc_request("session/new", new_params, 2),
                               ensure_ascii=False, separators=(",", ":")))
    resp = await client.call("session/new", new_params)
    show("←", resp)
    # sessionId 由服务端在响应里给出；后面 session/prompt / session/cancel 都要带它
    session_id = resp["result"]["sessionId"]

    # ---------- ④ 参数校验失败长什么样 ----------
    print("\n[4] 故意漏掉 cwd —— 看错误响应（JSON-RPC 错误码 -32602）")
    bad_params = {"mcpServers": []}
    print("    → " + json.dumps(rpc_request("session/new", bad_params, 3),
                               ensure_ascii=False, separators=(",", ":")))
    resp = await client.call("session/new", bad_params)
    show("←", resp)

    # ---------- ⑤ session/prompt：一轮对话 ----------
    print("\n[5] 发消息：session/prompt → 服务端先推一串 session/update 通知，最后才回响应")
    before = len(client.notifications)
    prompt_params = {
        "sessionId": session_id,
        "prompt": [{"type": "text", "text": "这个项目怎么跑起来？"}],
    }
    print("    → " + json.dumps(rpc_request("session/prompt", prompt_params, 4),
                               ensure_ascii=False, separators=(",", ":")))
    # 用 create_task 而不是 await：session/prompt 期间服务端会先推一串通知，
    # 必须让「等响应的协程」和「读循环」同时活着，通知才收得到
    prompt_task = asyncio.create_task(client.call("session/prompt", prompt_params))
    # 先歇一下让读循环把服务端推的通知全部取走，
    # 否则统计通知条数时可能「响应已经到了、通知还在路上」，看着像丢了消息
    await asyncio.sleep(0.05)
    resp = await prompt_task
    show("←", resp)
    # 数一下这一轮推了几条通知 —— 全部无 id，全部不需要客户端回复
    new_notifications = client.notifications[before:]
    print(f"    （这一轮共 {len(new_notifications)} 条 session/update 通知，全部无 id）")
    # 通知太多会刷屏，只挑头两条和最后一条展示（中间省略）
    for note in new_notifications[:2]:
        show("←", note)
    if len(new_notifications) > 3:
        print("    … 中间省略若干块 …")
    if new_notifications:
        show("←", new_notifications[-1])

    # ---------- ⑥ method 拼错 ----------
    print("\n[6] 方法名拼错：session/nwe → -32601 Method not found")
    print("    → " + json.dumps(rpc_request("session/nwe", {}, 5),
                               ensure_ascii=False, separators=(",", ":")))
    resp = await client.call("session/nwe", {})
    show("←", resp)

    # ---------- ⑦ session/cancel：通知 ----------
    print("\n[7] 取消：session/cancel —— 通知，服务端**不回复**")
    show("→", rpc_notification("session/cancel", {"sessionId": session_id}))
    await client.notify("session/cancel", {"sessionId": session_id})
    await asyncio.sleep(0.1)  # 给服务端一点处理时间
    print("    客户端收到的新消息数：0（0 = 果然一条都不回）")

    # 收尾：取消两个常驻任务并等它们真正退出（return_exceptions=True 避免取消异常往上冒）
    server_task.cancel()
    reader_task.cancel()
    await asyncio.gather(server_task, reader_task, return_exceptions=True)
    print()

下面这一格才是「跑」。分开写是为了让上面的定义格保持无输出，
也方便你把这一格反复重跑（它不依赖模型、不联网、不占端口）。

In [ ]:
# 源脚本这里是 asyncio.run(demo_memory_pipe())；内核里不能直接 asyncio.run，
# 所以改走 run_coro（脚本环境里它就是 asyncio.run），业务代码一个字没改
run_coro(demo_memory_pipe())

### 预期输出

本机实测（`cwd` 打印出来就是第 0 节 `os.chdir(ROOT)` 之后的仓库根，会随你的机器变）：

```text
==================================================================
演示 ① 内存管道：客户端协程 ↔ 服务端协程，完整 ACP 会话
==================================================================

[1] 批量请求：一个数组里塞两条，服务端回同样结构的数组
    （JSON-RPC 规范：批量请求可以混合成功与失败，逐条独立结算）
    → [{"jsonrpc":"2.0","id":101,"method":"initialize","params":{"protocolVersion":1}},{"jsonrpc":"2.0","id":102,"method":"session/nwe","params":{}}]
    ← [{"jsonrpc":"2.0","id":101,"result":{"protocolVersion":1,"agentCapabilities":{"loadSession":true,"promptCapabilities":{"embeddedContext":false,"image":false}}}},{"jsonrpc":"2.0","id":102,"error":{"code":-32601,"message":"Method not found：session/nwe"}}]

[2] 握手：initialize（交换协议版本与能力）
    → {"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":1,"clientCapabilities":{"fs":{"readTextFile":true,"writeTextFile":true}}}}
    ← {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":1,"agentCapabilities":{"loadSession":true,"promptCapabilities":{"embeddedContext":true,"image":false}}}}

[3] 建会话：session/new（必须给 cwd，否则 -32602）
    → {"jsonrpc":"2.0","id":2,"method":"session/new","params":{"cwd":"F:\\ProGram\\Python_Base","mcpServers":[]}}
    ← {"jsonrpc":"2.0","id":2,"result":{"sessionId":"sess_0001"}}

[4] 故意漏掉 cwd —— 看错误响应（JSON-RPC 错误码 -32602）
    → {"jsonrpc":"2.0","id":3,"method":"session/new","params":{"mcpServers":[]}}
    ← {"jsonrpc":"2.0","id":3,"error":{"code":-32602,"message":"Invalid params：session/new 必须带 cwd"}}

[5] 发消息：session/prompt → 服务端先推一串 session/update 通知，最后才回响应
    → {"jsonrpc":"2.0","id":4,"method":"session/prompt","params":{"sessionId":"sess_0001","prompt":[{"type":"text","text":"这个项目怎么跑起来？"}]}}
    ← {"jsonrpc":"2.0","id":4,"result":{"stopReason":"end_turn"}}
    （这一轮共 5 条 session/update 通知，全部无 id）
    ← {"jsonrpc":"2.0","method":"session/update","params":{"sessionId":"sess_0001","update":{"sessionUpdate":"agent_thought_chunk","content":{"type":"text","text":"（思考）学员问的是启动步骤…"}}}}
    ← {"jsonrpc":"2.0","method":"session/update","params":{"sessionId":"sess_0001","update":{"sessionUpdate":"agent_message_chunk","content":{"type":"text","text":"先执行 uv sync "}}}}
    … 中间省略若干块 …
    ← {"jsonrpc":"2.0","method":"session/update","params":{"sessionId":"sess_0001","update":{"sessionUpdate":"agent_message_chunk","content":{"type":"text","text":"动。"}}}}

[6] 方法名拼错：session/nwe → -32601 Method not found
    → {"jsonrpc":"2.0","id":5,"method":"session/nwe","params":{}}
    ← {"jsonrpc":"2.0","id":5,"error":{"code":-32601,"message":"Method not found：session/nwe"}}

[7] 取消：session/cancel —— 通知，服务端**不回复**
    → {"jsonrpc":"2.0","method":"session/cancel","params":{"sessionId":"sess_0001"}}
        [服务端] 收到 session/cancel 通知（无 id），停止会话 sess_0001
    客户端收到的新消息数：0（0 = 果然一条都不回）

```

**四处值得对照着看**：

1. `[1]` 的 `embeddedContext` 是 `false`，`[2]` 的是 `true` —— 因为 `[1]` 那条
   `initialize` 没带 `clientCapabilities`，服务端如实报告「对方没说自己能读文件」。
   这就是**能力协商**的实际影响，不是装饰；
2. `[1]` 的响应是**数组**，而且里面一条 `result` 一条 `error`
   —— JSON-RPC 允许批量请求里逐条独立结算；
3. `[4]` 走的是 `-32602 Invalid params`，而 `[6]` 走的是 `-32601 Method not found`
   —— 两个错误码别记混：前者是「方法对、参数不对」，后者是「压根没这个方法」；
4. `[7]` 客户端**收到的新消息数：0** —— 通知的定义就是「服务端不回复」。

### 3.7 演示 ②：真的起一个子进程，按行读写 JSON（真实 stdio）

演示 ① 的「传输层」是内存队列。真实世界的传输层是 **管道**：

```text
  编辑器（父进程）                      你的 ACP 服务端（子进程）
  proc.stdin.write(request + "\n")  →   for line in sys.stdin:  （一行一个 JSON）
  proc.stdout.readline()            ←   print(json.dumps(resp), flush=True)
```

下面这段字符串就是「你的 ACP 服务端」的最小等价物，用 `-c` 内联执行，
免得为了演示多建一个文件。它顺便踩出**两个真实世界才会遇到的坑**：

| 坑 | 后果 | 解法 |
|---|---|---|
| stdout 带缓冲 | 不 `flush` 对方会一直等（看起来像死锁） | 每条响应都 `flush=True` |
| 日志混进 stdout | **污染协议流**，对方解析失败 | 日志一律走 `stderr` |

下面第一格先定义子进程脚本（`CHILD_SERVER_SCRIPT`），第二格才是父进程侧。

In [ ]:
import subprocess

CHILD_SERVER_SCRIPT = r'''
import json, sys

# 子进程也要显式指定编码，否则 Windows 下默认 GBK 会读不动 UTF-8 请求
sys.stdin.reconfigure(encoding="utf-8")
sys.stdout.reconfigure(encoding="utf-8")

sessions = {}
for line in sys.stdin:                      # ← 一行一个完整 JSON 对象
    line = line.strip()
    if not line:
        continue
    try:
        msg = json.loads(line)              # ← 解帧：一行 = 一条消息
    except json.JSONDecodeError:
        sys.stdout.write(json.dumps({"jsonrpc": "2.0", "id": None,
            "error": {"code": -32700, "message": "Parse error"}}) + "\n")
        sys.stdout.flush()
        continue

    method = msg.get("method")
    params = msg.get("params") or {}
    rid = msg.get("id")
    if rid is None:                         # 通知：不回复
        print("child: got notification " + str(method), file=sys.stderr, flush=True)
        continue

    if method == "initialize":
        result = {"protocolVersion": 1, "agentCapabilities": {"loadSession": True}}
    elif method == "session/new":
        sid = "sess_child_1"
        sessions[sid] = params.get("cwd")
        result = {"sessionId": sid}
    elif method == "session/prompt":
        result = {"stopReason": "end_turn"}
    else:
        print(json.dumps({"jsonrpc": "2.0", "id": rid,
            "error": {"code": -32601, "message": "Method not found: " + str(method)}}),
            flush=True)
        continue

    # flush=True 是关键：不加它，管道缓冲会让父进程永远读不到这一行
    print(json.dumps({"jsonrpc": "2.0", "id": rid, "result": result}, ensure_ascii=False),
          flush=True)
'''

父进程侧 `demo_subprocess_pipe()` —— 这就是 **PyCharm / Zed 启动你 ACP 服务端脚本时做的事**：

1. `subprocess.Popen([sys.executable, "-c", CHILD_SERVER_SCRIPT], stdin=PIPE, stdout=PIPE, stderr=PIPE)`
   —— 三根管子各司其职：**stdin/stdout 跑协议，stderr 走日志**；
2. `send()`：把报文序列化成一行（`separators` 去空格，就是线路上的紧凑形式）+ `\n`，
   然后 **`flush()`**（不 flush，子进程会一直阻塞在 `readline` 上）；
3. `recv()`：从子进程 stdout **读一行**，`json.loads` 解出来；
4. 依次发 `initialize` / `session/new` / `session/prompt`（各收一条同 id 响应）；
5. 发一条 `session/cancel` **通知** —— 子进程不回，只在 stderr 记一行日志；
6. 故意发一条**截断的坏 JSON** —— 子进程解析失败，回 `-32700`，
   而且此时 `id` 是 **`null`**（它根本解析不出你用的是哪个 id）；
7. `proc.stdin.close()` 让子进程的 `for` 循环自然结束 → `wait()` 收尸 → 读 stderr 日志。

`finally` 里那句 `proc.kill()` 是**安全网**：无论上面哪一步出错，
都不能把子进程留在后台。

In [ ]:
def demo_subprocess_pipe() -> None:
    """演示 ②：subprocess + stdin/stdout 真实收发 JSON-RPC。"""
    print("=" * 66)
    print("演示 ② 真实子进程：subprocess + stdin/stdout（PyCharm 就是这么起你的脚本）")
    print("=" * 66)

    env = dict(os.environ)
    # PYTHONIOENCODING：强制子进程 stdio 用 UTF-8，避开 Windows 默认 GBK
    env["PYTHONIOENCODING"] = "utf-8"

    # sys.executable = 当前解释器（本项目的 .venv），不用去猜 python 在哪；
    # stdin/stdout 都用 PIPE —— 这正是「stdio 传输」的字面含义：协议跑在管道上
    proc = subprocess.Popen(
        [sys.executable, "-c", CHILD_SERVER_SCRIPT],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,   # stderr 单独开一根管子，日志才不会污染协议流
        text=True,
        encoding="utf-8",
        env=env,
    )

    def send(message: dict) -> None:
        """往子进程 stdin 写一行 JSON（带 \\n，行分隔协议）。"""
        # separators 去空格：线路上传输的就是这种紧凑形式
        line = json.dumps(message, ensure_ascii=False, separators=(",", ":"))
        print(f"    → {line}")
        assert proc.stdin is not None
        proc.stdin.write(line + "\n")
        proc.stdin.flush()  # 不 flush 子进程会一直阻塞在 readline 上

    def recv() -> dict:
        """从子进程 stdout 读一行 JSON。"""
        assert proc.stdout is not None
        line = proc.stdout.readline()
        print(f"    ← {line.strip()}")
        return json.loads(line)

    try:
        # ① 握手：按「请求 → 必须收到一条同 id 响应」的规则走
        send(rpc_request("initialize", {"protocolVersion": 1, "clientCapabilities": {}}, 1))
        recv()

        # ② 建会话：id=2，同样等一条响应
        send(rpc_request("session/new", {"cwd": os.getcwd(), "mcpServers": []}, 2))
        recv()

        # ③ 发消息：真实服务端会在响应之前推一串 session/update 通知，
        #    这里的子进程脚本简化了，直接回一条 stopReason 响应
        send(rpc_request("session/prompt", {"sessionId": "sess_child_1",
                                            "prompt": [{"type": "text", "text": "hi"}]}, 3))
        recv()

        # 通知：发出去后子进程不会回（它只往 stderr 记了一行日志）
        note = rpc_notification("session/cancel", {"sessionId": "sess_child_1"})
        print(f"    → {json.dumps(note, ensure_ascii=False, separators=(',', ':'))}")
        assert proc.stdin is not None
        proc.stdin.write(json.dumps(note) + "\n")
        proc.stdin.flush()
        print("    ← （无响应 —— 通知的定义就是服务端不回复）")

        # 故意发一条坏 JSON，验证 -32700 Parse error 也是「一行一条响应」
        # （注意此时服务端根本解析不出 id，所以它回的那条 error 里 id 是 null）
        print('    → {"jsonrpc":"2.0","id":9,"method":  <-- 故意截断的坏 JSON')
        assert proc.stdin is not None
        proc.stdin.write('{"jsonrpc":"2.0","id":9,"method":\n')
        proc.stdin.flush()
        recv()

        # 收尾：关掉 stdin 让子进程的 for 循环自然结束，再等它退出并收 stderr 日志
        assert proc.stdin is not None
        proc.stdin.close()  # 关掉 stdin → 子进程 for 循环结束，正常退出
        proc.wait(timeout=10)
        stderr = proc.stderr.read() if proc.stderr else ""
        print(f"    子进程退出码：{proc.returncode}")
        print("    子进程 stderr（日志走这里，不污染协议流）：")
        for line in stderr.strip().splitlines():
            print(f"        {line}")
    finally:
        # 无论上面哪一步出错，都不能把子进程留在后台
        if proc.poll() is None:
            proc.kill()
    print()

跑它。这一格**只需要标准库**：不装 `acp`、不连模型、不占端口，纯靠管道通一次话。

In [ ]:
demo_subprocess_pipe()

### 预期输出

```text
==================================================================
演示 ② 真实子进程：subprocess + stdin/stdout（PyCharm 就是这么起你的脚本）
==================================================================
    → {"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":1,"clientCapabilities":{}}}
    ← {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": 1, "agentCapabilities": {"loadSession": true}}}
    → {"jsonrpc":"2.0","id":2,"method":"session/new","params":{"cwd":"F:\\ProGram\\Python_Base","mcpServers":[]}}
    ← {"jsonrpc": "2.0", "id": 2, "result": {"sessionId": "sess_child_1"}}
    → {"jsonrpc":"2.0","id":3,"method":"session/prompt","params":{"sessionId":"sess_child_1","prompt":[{"type":"text","text":"hi"}]}}
    ← {"jsonrpc": "2.0", "id": 3, "result": {"stopReason": "end_turn"}}
    → {"jsonrpc":"2.0","method":"session/cancel","params":{"sessionId":"sess_child_1"}}
    ← （无响应 —— 通知的定义就是服务端不回复）
    → {"jsonrpc":"2.0","id":9,"method":  <-- 故意截断的坏 JSON
    ← {"jsonrpc": "2.0", "id": null, "error": {"code": -32700, "message": "Parse error"}}
    子进程退出码：0
    子进程 stderr（日志走这里，不污染协议流）：
        child: got notification session/cancel
```

**四行要仔细看**：

1. `→` 的行是**紧凑**的（`separators` 去掉了空格），`←` 的行带空格 ——
   因为子进程脚本用的是普通 `json.dumps()`。**两种都合法**：
   JSON 的空白不影响解析，行分隔协议只要求「一行一个完整对象」；
2. `session/cancel` 发出后**没有任何 `←` 行**，但最后 stderr 里出现了
   `child: got notification session/cancel` —— 这一对就是「通知不回复」的完整证据；
3. 坏 JSON 换回来的是 `"id": null` 的 `-32700` —— 因为服务端**根本解析不出**
   你用的是哪个 id（第 3.1 节讲的那个边界情况）；
4. `子进程退出码：0` —— 关掉 stdin 让 `for line in sys.stdin` 自然结束，
   这是「优雅收尾」的标准做法，不要用 `kill` 收尾（`kill` 只是 `finally` 的安全网）。

### 3.8 源脚本的入口（对照用）

源文件 `02_acp原理_jxsd.py` 的末尾是一个 `main()`：

```text
asyncio.run(demo_memory_pipe())   ← 演示 ①
demo_subprocess_pipe()            ← 演示 ②
```

notebook 里这两步**已经各自单独跑过**（3.6 / 3.7），所以下面只保留这个入口函数的
定义做对照，**不再调用它** —— 否则两个演示会白白重复跑一遍。
定义里那句 `asyncio.run(demo_memory_pipe())` 也原样留着，方便你对照
「脚本里怎么写」与「notebook 里为什么得改成 `run_coro(...)`」。

In [ ]:
def main_jsonrpc() -> None:
    print("ACP 原理：JSON-RPC 2.0 的真实收发")
    print()

    # 演示 ① 用 asyncio（协议本身是异步双向流，同步写法做不出「一边等响应一边收通知」）
    asyncio.run(demo_memory_pipe())

    # 演示 ② 用同步 subprocess（这里要让学员看清「一行一行读」的手感）
    demo_subprocess_pipe()

    # 小结把「判定规则」再钉一遍 —— 这三条是本节唯一需要背下来的东西
    print("=" * 66)
    print("小结：ACP 没有魔法 —— 就是「一行一个 JSON 对象」跑在 stdin/stdout 上。")
    print("     · 有 id 的必回一条同 id 的响应（result 或 error）")
    print("     · 无 id 的是通知，服务端不回复（流式输出全靠它）")
    print("     · 服务端也能反向发请求（fs/read_text_file、session/request_permission）")
    print("     deepagents-acp 的 AgentServerACP + run_agent() 帮你把上面这些全做了，")
    print("     你只写 create_deep_agent(...)（见 01_acp智能体_jxsd.py）。")
    print("=" * 66)

## 小结

- **ACP 是「编辑器 ↔ 智能体」的插座标准**，类比 LSP；它**不是** Agent 之间通信
  （那是 A2A），也**不是** Agent 调工具（那是 MCP）；
- **两行代码就能插上插座**：`AgentServerACP(agent)` + `await run_agent(server)`，
  默认走 **stdio**（编辑器起子进程，stdin 读请求 / stdout 写响应）；
- **编辑器侧只配一个 `acp.json`**：`command`（解释器绝对路径）+ `args`（脚本路径）——
  用错解释器是**静默失败**，这是最常见的坑；
- **协议本体是 JSON-RPC 2.0**，只有两条规则：**有 id 必回一条同 id 响应**、
  **无 id 的是通知**；传输层是「**一行一个 JSON 对象**」；
- **ACP 的核心方法 6 个**：`initialize` / `session/new` / `session/load` /
  `session/prompt` / `session/cancel`（通知）/ `session/update`（通知）；
- **顺序很关键**：`session/prompt` 期间先推完所有 `session/update`，
  **最后**才回带 `id` 的响应（带 `stopReason`）；
- **流式输出没有魔法**：它就是「一条接一条、没有 id 的 `session/update`」。

下一课 `02_A2A协议.ipynb` 会把这里的 JSON-RPC 报文原样搬到 **HTTP** 上 ——
那时你会看到：**换的只是传输层，报文长得一模一样**。

## 常见坑

1. **解释器用错 → 静默失败**。编辑器会以「`ModuleNotFoundError: No module named 'acp'`」
   直接退出，界面上什么都不显示。`acp.json` 的 `command` 必须是**装了
   `deepagents-acp` 的那个解释器**的绝对路径（Windows 记得带 `.exe`）。
2. **Windows 路径在 JSON 里要写双反斜杠**。`"C:\\Users\\..."`；
   写成单反斜杠的 `\U` 会被 JSON 解析器报 `Invalid \escape`。
3. **`asyncio.run()` 在 notebook 内核里必失败**（本机实测报
   `RuntimeError: asyncio.run() cannot be called from a running event loop`），
   因为 ipykernel 一直在跑自己的事件循环。本课用 `run_coro()` 兜住；
   你自己写协程演示时也要注意 —— 在 `.py` 脚本里 `asyncio.run` 才是对的。
4. **调试 `print` 污染协议流**。`run_agent(server)` 之后 stdout **属于协议**，
   任何 `print` 都会让编辑器解析失败。日志一律走 `stderr`（3.7 小节的子进程演示
   就是标准做法）。
5. **忘了 `flush`**。管道带缓冲，响应不 `flush=True` 对方会一直等，
   现象是「看起来像死锁、其实是没发出去」。
6. **该回的不回、不该回的乱回**。判据只有一个：**有没有 `id`**。
   给通知回一条响应，就是协议错误。
7. **`session/prompt` 的响应回早了**。必须先把所有 `session/update` 推完，
   最后才回带 `id` 的响应；提前回会让客户端认为本轮已结束。
8. **`-32601` 与 `-32602` 别记混**：前者是「**没这个方法**」（方法名拼错），
   后者是「**方法对、参数不对**」（例如 `session/new` 没带 `cwd`）。
9. **`jsonrpc` 是字符串 `"2.0"`，不是数字 `2.0`**。写成数字会被
   `_dispatch()` 的第一道校验判成 `-32600 Invalid Request` —— 这类错误
   报的是「请求结构不对」，而不是「方法找不到」，看错误码能直接定位。
10. **装了 `deepagents-acp` 之后再跑第 2.6 节会阻塞** —— 那时它是真的在等编辑器
    发 `initialize`。请在 notebook 里手动中断内核，不要去改代码。

## 官方链接

ACP 协议本体：

- ACP 官方站 · 介绍：<https://agentclientprotocol.com/get-started/introduction>
- ACP 协议 v1 · Overview（消息流 / 方法 / 通知）：<https://agentclientprotocol.com/protocol/v1/overview>
- ACP 支持的客户端一览：<https://agentclientprotocol.com/get-started/clients>
- ACP 官方仓库：<https://github.com/agentclientprotocol/agent-client-protocol>

接法与编辑器配置：

- LangChain 文档 · Deep Agents 的 ACP 接入（安装 + `AgentServerACP` + 编辑器配置）：<https://docs.langchain.com/oss/python/deepagents/acp>
- Zed 外部 Agent 文档：<https://zed.dev/docs/ai/external-agents>
- JetBrains AI Assistant 的 ACP 支持（课案里 `~/.jetbrains/acp.json` 的来源）：<https://www.jetbrains.com/help/ai-assistant/acp.html>
- VS Code 的 `vscode-acp` 插件：<https://github.com/formulahendry/vscode-acp>
- `deepagents` 仓库里可直接注册进 Zed 的 demo 入口：<https://github.com/langchain-ai/deepagents/blob/main/libs/acp/run_demo_agent.sh>

报文格式：

- JSON-RPC 2.0 规范（四种形态与错误码的权威出处）：<https://www.jsonrpc.org/specification>

本仓相邻内容：

- MCP（智能体 ↔ 工具服务）：`Agent/05_mcp/`
- A2A（智能体 ↔ 智能体，JSON-RPC over HTTP）：`Agent/07_protocols/02_A2A协议.ipynb`